In [ ]:
import pandas as pd
from Bio import SeqIO
import sys
import os
from ast import literal_eval
import numpy as np


import pybedtools
from pybedtools import BedTool


#For plotting
from matplotlib.colors import LinearSegmentedColormap
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

#For statistics
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import itertools

import re
from Bio import SeqIO
import ast # for safe eveal, for parsing some of the data
import math
# --- Shared helpers: const.py lives in analyses/common ---
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'common')))

#import importlib
#importlib.reload(const)

import const #to reload use import(importlib) and then importlib.reload(const)
#from const import pos_active_ctrl_color,neg_active_ctrl_color,highlight_color,custom_cmap
from const import set_equal_plot_limits
from const import plot_color_pallete
from const import custom_cmap_bolder
from const import FONT_SIZE_small
const.set_plot_style()
import matplotlib.ticker as mtick

# --- Working directory (EDIT): used for relative .bed/.csv I/O in some notebooks ---
WORK_DIR = '/home/labs/davidgo/Collaboration/backup/humanMPRA/scripts/produce_paper_figures'
os.chdir(WORK_DIR)

output_path = '/home/labs/davidgo/Collaboration/humanMPRA/paper/raw_plots/fig_library_and_QC/'

# Parameters

In [ ]:
# Use CPM normalization?
cpm=True

# Use log scale in visualization?
logScale=False

#Additional filters?

min_DNA_counts = 50

# Figure 1 - hMPRA validation

### import and process all libraries

In [ ]:
full_activity_df = pd.read_csv('/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes/quantitative_analysis_combined/comb_df_combined_fdr.csv')


#### Apply filters

In [ ]:
# Keep only rows where the 'oligo' column contains "SCREEN"
print(f"oligos before filtering for SCREEN overlap:{len(full_activity_df)}")
full_activity_df = full_activity_df[full_activity_df["oligo"].str.contains("SCREEN", na=False)]
print(f"oligos after filtering for SCREEN overlap:{len(full_activity_df)}")
full_activity_df =  full_activity_df[~full_activity_df["oligo"].str.contains("Ctrl", na=False)]
print(f"oligos after filtering for non-controls:{len(full_activity_df)}")


if min_DNA_counts>0:
    full_activity_df =  full_activity_df[(full_activity_df['DNA_rep_comb']>min_DNA_counts)]
    print(f"oligos after filtering for oligos with over {min_DNA_counts} DNA counts (combined for all replicates):{len(full_activity_df)}")



# remove oligos which were later removed
full_activity_df = full_activity_df[~(full_activity_df['orientation_fix']=='fixed_in_L4')&
                           ~(full_activity_df['oligo'].str.contains('SCREEN_EH'))&
                          ~(full_activity_df['oligo'].str.contains('hh.missing.oligos')) ]
print(f"oligos after filtering out oligos which were later remoed due to L4a1 (eg orientation fix):{len(full_activity_df)}")

#if min_barcodes>0:
#    full_activity_df =  full_activity_df[(full_activity_df['bcs_DNA_rep1']>10) &(full_activity_df['bcs_DNA_rep2']>10)]
#    print(f"oligos after filtering for oligos with over 10 barcodesin both reps 1 and 2:{len(full_activity_df)}")


In [ ]:
# Add normalization of RNA and DNA counts
DNA_sum = full_activity_df["DNA_rep_comb"].sum()
RNA_sum = full_activity_df["RNA_rep_comb"].sum()

full_activity_df["DNA_rep_comb_cpm"] = 1000000*(full_activity_df["DNA_rep_comb"]+1)/DNA_sum
full_activity_df["RNA_rep_comb_cpm"] = 1000000*(full_activity_df["RNA_rep_comb"]+1)/RNA_sum

full_activity_df["DNA_rep_comb_cpm_log"] = np.log2(full_activity_df["DNA_rep_comb_cpm"])
full_activity_df["RNA_rep_comb_cpm_log"] = np.log2(full_activity_df["RNA_rep_comb_cpm"])


In [ ]:
full_comparative_df = pd.read_csv("/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes/comparative_analysis_combined/mpranalyze_comp_res_fdr_complete.csv")
full_comparative_df = full_comparative_df[['oligo','logFC','differntial.wo_controls']]
full_comparative_df = full_comparative_df.rename(
    columns={'differntial.wo_controls': 'differential_activity'}
)


## Drop shape

In [ ]:
# Prepare the data
# Replace Inf values with NaN, then drop any rows with NaN values
replicates_activity_df = full_activity_df[['oligo','ratio_log_rep1','ratio_log_rep2','bcs_DNA_rep1','bcs_DNA_rep2']].copy()

print(f'Number of oligos: {len(replicates_activity_df)}')

replicates_activity_df.replace([np.inf, -np.inf], np.nan)

# Drop rows where either 'ratio_log_filtered_std2_rep1' or 'ratio_log_filtered_std2_rep2' has NaN or Inf
replicates_activity_df = replicates_activity_df.dropna(subset=['ratio_log_rep1', 'ratio_log_rep2'])

#replicates_activity_df = replicates_activity_df.sample(n=60000, random_state=42)

x = replicates_activity_df['ratio_log_rep1'].values
y = replicates_activity_df['ratio_log_rep2'].values


In [ ]:
colors = ["#EBF4FF", "#E1ECFA", "#D0E0F5", "#FFFFBF", "#FEE090", "#FDAE61", "#F46D43", "#D73027", "#A50026"]
custom_cmap_light = LinearSegmentedColormap.from_list("custom_cmap", colors, N=256)

In [ ]:
plt.clf()

fig, ax = plt.subplots()

hb = ax.hexbin(
    x, y,
    gridsize=100,          # adjust for coarser/finer grid
    cmap=custom_cmap_light,
    mincnt=1,             # only show hexes with at least 1 point
    bins='log',            # color ~ log(count); remove if you want raw counts
    linewidths=0
    
)

# ---- Pearson correlation between x and y ----
r = np.corrcoef(x, y)[0, 1]

# Add correlation text inside the axes (top-left corner)
ax.text(
    0.02, 0.98,
    rf'$r = {r:.2f}$',
    transform=ax.transAxes,
    ha='left', va='top'
)


ax.set_xlabel(r'$\log_{2}\!\left(\frac{\mathrm{RNA}}{\mathrm{DNA}}\right)$ replicate 1')
ax.set_ylabel(r'$\log_{2}\!\left(\frac{\mathrm{RNA}}{\mathrm{DNA}}\right)$ replicate 2')

# Keep your tick simplification
xticks = ax.get_xticks()
yticks = ax.get_yticks()
ax.set_xticks([xticks[0], xticks[-1]])
ax.set_yticks([yticks[0], yticks[-1]])

ax.set_aspect('equal', adjustable='box')
# Optional: set explicit limits if you want
# ax.set_xlim(-4, 6)
# ax.set_ylim(-4, 6)

const.save_fig(plt, f'RNA_DNA_ratio_correlation_between_replicates_min_dna_{min_DNA_counts}', output_path)

# Add colorbar
cb = fig.colorbar(hb, ax=ax)
cb.set_label(r'Density ($\log_{10}$)')

const.save_fig(plt, f'RNA_DNA_ratio_correlation_between_replicates_min_dna_{min_DNA_counts}_w_bar', output_path)
plt.show()


## Activity histogram

In [ ]:
bin_edges = np.linspace(-4, 4, 201)  # 100 bins between -10 and 20

# First histogram (all scores)
plt.hist(
    data=full_activity_df,
    x='ratio_log_rep_comb', bins=bin_edges, color=plot_color_pallete['default_color'], label='Non-active cCREs')

# Second histogram (filtered scores)
plt.hist(
    data=full_activity_df[full_activity_df['activity_adjusted'] == 'active'],
    x='ratio_log_rep_comb', bins=bin_edges, color='red', label='Active cCREs')

plt.xlabel(r'$\log_{2}\!\left(\frac{\mathrm{RNA}}{\mathrm{DNA}}\right)$')
plt.ylabel('Number of cCREs')
plt.legend()
stat,pval=stats.skewtest(full_activity_df['ratio_log_rep_comb'].dropna())

# Smallest positive float in Python
min_p = np.nextafter(0, 1)  # ~5e-324 for float64

if pval < min_p:
    p_text = f"p < {min_p:.1e}"
else:
    p_text = f"p = {pval:.3g}"

plt.text(
    0.05, 0.95,  # relative position in axes coordinates
    f"Skew test:\nstat = {stat:.2f}\n{p_text}",
    ha='left', va='top',
    transform=plt.gca().transAxes,
    fontsize=10,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8)
)


plt.xticks([-4,0,4])

yticks = plt.gca().get_yticks()
plt.yticks([yticks[0], yticks[-1]])

plt.legend(
    loc="upper right",
    fontsize=10,         # smaller text
    markerscale=0.8,    # shrink the color boxes
    handlelength=1,     # shorter legend lines
    handletextpad=0.5,  # less padding between box and text
    borderpad=0.3       # tighter box
)


const.save_fig(plt,f'activity_distribution_min_dna_{min_DNA_counts}',output_path)
plt.show()


##  Controls boxplot


In [ ]:
# List of libraries
libraries = [
    "L1a1", "L1a2", "L1a3",
    "L2a1", "L2a2", "L2a3",
    "L3a1", "L3a2", "L3a3",
    "L4a1"
]

base_dir = r"/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes"

dfs = []

for lib in libraries:
    csv_path = os.path.join(
        base_dir,
        lib,
        "output",
        "activity_after_filter",
        "comb_df_adjusted_fdr.csv"
    )

    # Read CSV
    df = pd.read_csv(csv_path)

    # Keep only rows where "oligo" contains "ctrl"
    df_ctrl = df[
    df["oligo"].str.contains("Ctrl", na=False, case=False) |
    df["oligo"].str.contains("scramble", na=False, case=False)
        ].copy()
    
    # Optional: add a column to remember which library it came from
    df_ctrl["library"] = lib

    dfs.append(df_ctrl)

# Combine all control oligos from all libraries into one df
all_ctrl_oligos = pd.concat(dfs, ignore_index=True)


In [ ]:
control_types = [
    "NegCtrl_active_not_diff_ESCs+Osteoblasts+NPCs",
    "NegCtrl_non_SCREEN",
    "scrambled_control",
    "PosCtrl_diff_ESCs_Weiss_seq11687_derived_a1_L4",
    "NegCtrl_not_active_ESCs+Osteoblasts+NPCs",
    "PosCtrl_diff_ESCs+Osteoblasts+NPCs",
    "PosCtrl_chondrocyte_active",
    "PosCtrl_diff_NPCs_Weiss",
    "PosCtrl_diff_ESCs+NPCs",
    "PosCtrl_diff_ESCs_Weiss",
    "PosCtrl_diff_ESCs+Osteoblasts",
    "PosCtrl_diff_Osteoblasts+NPCs",
    "PosCtrl_diff_Osteoblasts",
    "PosCtrl_neuron_active",
    "PosCtrl_osteoblast_active",
]

ctrl_dict = {}

for ctype in control_types:
    mask = all_ctrl_oligos["oligo"].str.contains(ctype, regex=False)
    ctrl_dict[ctype] = all_ctrl_oligos[mask].copy()

# Optional: if you want a df of any remaining controls that didn't match:
unmatched_mask = ~pd.concat(
    [all_ctrl_oligos["oligo"].str.contains(ct, na=False, regex=False) for ct in control_types],
    axis=1
).any(axis=1)

ctrl_dict["other_ctrls"] = all_ctrl_oligos[unmatched_mask].copy()

to_combine = [
    "PosCtrl_chondrocyte_active",
    "PosCtrl_osteoblast_active",
    "NegCtrl_active_not_diff_ESCs+Osteoblasts+NPCs"
]

ctrl_dict["all_activity_pos_ctrl"] = (
    pd.concat([ctrl_dict[k] for k in to_combine], ignore_index=True)
)

#Add the tested cCREs
# Add tested cCREs as a new entry in the control dictionary
# (here we only keep mad.score since that's what we plot)

#Define a "pair_id" by removing the ancestral/derived part from the oligo name
def make_pair_id(s):
    # remove '_ancestral', '-ancestral', ' ancestral', etc. (same for derived)
    s = re.sub(r"[_\- ]?(ancestral|derived)", "", s, flags=re.IGNORECASE)
    return s

tested_cCREs = full_activity_df[["oligo","mad.score","activity"]].copy()
tested_cCREs["pair_id"] = tested_cCREs["oligo"].apply(make_pair_id)
#tested_cCREs["mad.score"] = 

ctrl_dict["tested_cCREs"] = tested_cCREs[["pair_id","mad.score","activity",]]




In [ ]:
tested_cCREs

In [ ]:
pos_color  = '#8FBCE1'
neg_color  = '#DE2326'
high_color = '#BBA9D2'

# Keys from the dictionary and their pretty labels
plot_keys = {
    "scrambled_control": "Negative control (scrambled)",
    "NegCtrl_non_SCREEN": "Negative control (non-SCREEN)",
    "all_activity_pos_ctrl": "Positive control (active)",
    "NegCtrl_not_active_ESCs+Osteoblasts+NPCs": "Negative control (inactive)",
    "tested_cCREs_active": "cCREs (active)",
    "tested_cCREs_inactive": "cCREs (non-active)",
}

plot_dfs = []
for key, pretty in plot_keys.items():
    if key == "tested_cCREs_active":
        df_temp = ctrl_dict["tested_cCREs"][ctrl_dict["tested_cCREs"]['activity'] == 'active'].copy()
    elif key == "tested_cCREs_inactive":
        df_temp = ctrl_dict["tested_cCREs"][ctrl_dict["tested_cCREs"]['activity'] == 'non_active'].copy()
    elif key not in ctrl_dict:
        continue
    else:
        df_temp = ctrl_dict[key].copy()
    
    df_temp["control_type"] = key
    df_temp["control_type_pretty"] = pretty
    plot_dfs.append(df_temp)

c = pd.concat(plot_dfs, ignore_index=True)

# Update order and palette for the new categories
order = [
    "Positive control (active)",
    "Negative control (inactive)",
    "Negative control (scrambled)",
    "Negative control (non-SCREEN)",
    "cCREs (active)",
    "cCREs (non-active)",
]

palette = [
    '#90EE90',   # Positive control (active)
    plot_color_pallete['default_color'],   # Negative control (inactive)
   plot_color_pallete['default_color'],
    plot_color_pallete['default_color'],
    'red',   # cCREs (active) - light green
    plot_color_pallete['default_color']
]

sns.boxplot(
    data=c,
    x="mad.score",
    y="control_type_pretty",
    showfliers=True,
    linewidth=2,
    order=order,
    palette=palette
)
plt.xlim(-10,30)

plt.ylabel("")
plt.xlabel("Activity statistic")

const.save_fig(plt,f'activity_distribution_boxplots_min_dna_{min_DNA_counts}',output_path)

plt.show()

In [ ]:
c_capped = c[c['mad.score']<30]
sns.violinplot(
    data=c_capped,
    x="mad.score",
    y="control_type_pretty",
    #showfliers=False,
    linewidth=2,
    order=order,
    palette=palette
)
plt.ylabel("")
plt.xlabel(f"Activity statistic_min_dna_{min_DNA_counts}")

In [ ]:
#4. Merge with differential activity df 
tested_cCREs = tested_cCREs.merge(
    full_comparative_df,
    how='left',
    left_on='pair_id',
    right_on='oligo'
)

In [ ]:
# Compare tested cCREs across: active, differential active, and non-active
tested_compare_df = tested_cCREs.copy()

# Keep valid activity score rows
tested_compare_df = tested_compare_df[
    tested_compare_df["mad.score"].notna() &
    np.isfinite(tested_compare_df["mad.score"])
].copy()

# Build mutually exclusive groups:
# 1) differential active (from comparative analysis)
# 2) active (but not differential active)
# 3) non-active
tested_compare_df.loc[tested_compare_df["differential_activity"] == True, "group"] = "Differential active"
tested_compare_df.loc[
    (tested_compare_df["activity"] == "active") & (~tested_compare_df["differential_activity"].eq(True)),
    "group",
] = "Active (not differential)"
tested_compare_df.loc[tested_compare_df["activity"] == "non_active", "group"] = "Non-active"

tested_compare_df = tested_compare_df.dropna(subset=["group"]).copy()

order_groups = ["Differential active", "Active (not differential)", "Non-active"]
palette_groups = ["#D73027", "#FC8D59", "#AEAEAE"]

plt.figure(figsize=(4, 5))
sns.boxplot(
    data=tested_compare_df,
    x="mad.score",
    y="group",
    order=order_groups,
    palette=palette_groups,
    linewidth=2,
    showfliers=True
)

plt.xlabel("Activity statistic (MAD)")
plt.ylabel("")
plt.xlim(-10, 20)

const.save_fig(plt, f"tested_cCREs_active_vs_diff_vs_nonactive_min_dna_{min_DNA_counts}", output_path)
plt.show()

# Pairwise KS tests
print("Pairwise KS tests:")
for g1, g2 in itertools.combinations(order_groups, 2):
    v1 = tested_compare_df.loc[tested_compare_df["group"] == g1, "mad.score"].to_numpy()
    v2 = tested_compare_df.loc[tested_compare_df["group"] == g2, "mad.score"].to_numpy()
    ks = stats.ks_2samp(v1, v2)
    print(f"{g1} vs {g2}: D = {ks.statistic:.4f}, p = {ks.pvalue:.3e}, n1 = {len(v1)}, n2 = {len(v2)}")


## RNA vs DNA

In [ ]:
# Prepare the data
x = full_activity_df['DNA_rep_comb'].values
y = full_activity_df['RNA_rep_comb'].values

# Build a mask removing NaN and ±inf in both x and y
mask = (
    np.isfinite(x) &
    np.isfinite(y) &
    (y < 300_000)
)

# Apply mask
x = x[mask]
y = y[mask]

print(len(x), "points remain after filtering.")


# Evaluate the KDE on the grid
#values = np.vstack([x, y])
#kernel = gaussian_kde(values)
#density = kernel(values)

In [ ]:
plt.clf()
fig, ax = plt.subplots()

# Hexbin density plot instead of scatter
hb = ax.hexbin(
    x, y,
    gridsize=100,              # tweak for smoother/coarser bins
    cmap=custom_cmap_bolder,  # same colormap as before
    mincnt=1,                 # only show bins with at least 1 point
    bins='log',                # log10 of counts
    linewidth = 0
)

# Get current axis limits
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()

m = sum(y)/sum(x)
# Define x-range for the line
x_line = np.array([xmin, xmax])
y_line = m * x_line

ax.plot(
    x_line,
    y_line,
    color='black',
    linestyle='--',
    linewidth=1.2,
    alpha=0.8,
    label=f'y = {m:.3g}·x'
)



# Set axis limits
ax.set_xlim(0, max(x))
ax.set_ylim(0, max(y))

# Simplify ticks (first & last)
xticks = ax.get_xticks()
yticks = ax.get_yticks()
ax.set_xticks([xticks[0], xticks[-1]])
ax.set_yticks([yticks[0], yticks[-1]])

# Axis labels
ax.set_xlabel('DNA (CPM)')
ax.set_ylabel('RNA (CPM)')

const.save_fig(plt, f'RNA_DNA_ratio_min_dna_{min_DNA_counts}', output_path)

# Colorbar with label
cb = fig.colorbar(hb, ax=ax)
cb.set_label(r'Density ($\log_{10}$)')

const.save_fig(plt, f'RNA_DNA_ratio_min_dna_{min_DNA_counts}_w_bar', output_path)

plt.show()


## ancestral vs derived alleles

In [ ]:
ancestral_derived_data = full_activity_df.copy()
print(len(ancestral_derived_data))
ancestral_derived_data = ancestral_derived_data[ancestral_derived_data['count_rep_comb']>min_DNA_counts]
#ancestral_derived_data = ancestral_derived_data[ancestral_derived_data['activity_adjusted_combined']=='active']
print(len(ancestral_derived_data))

# 1. Keep only rows that are ancestral/derived
ancestral_derived_data = ancestral_derived_data[ancestral_derived_data["oligo"].str.contains("ancestral|derived", case=False, na=False)].copy()


# 2. Add a column indicating whether it's ancestral or derived
ancestral_derived_data["status"] = np.where(
    ancestral_derived_data["oligo"].str.contains("ancestral", case=False, na=False),
    "ancestral",
    "derived"
)

# 3. Define a "pair_id" by removing the ancestral/derived part from the oligo name
def make_pair_id(s):
    # remove '_ancestral', '-ancestral', ' ancestral', etc. (same for derived)
    s = re.sub(r"[_\- ]?(ancestral|derived)", "", s, flags=re.IGNORECASE)
    return s

ancestral_derived_data["pair_id"] = ancestral_derived_data["oligo"].apply(make_pair_id)


# 4. Pivot so each row = one pair, with ancestral / derived mad.score as columns
ancestral_derived_data = (
    ancestral_derived_data
    .pivot_table(
        index="pair_id",
        columns="status",
        values="mad.score",
        aggfunc="first"   # in case there are duplicates
    )
    .reset_index()
)

#4. Merge with differential activity df 
ancestral_derived_data = ancestral_derived_data.merge(
    full_comparative_df,
    how='left',
    left_on='pair_id',
    right_on='oligo'
)



# Columns are now: ['pair_id', 'ancestral', 'derived']
# 5. Filter to keep only pairs where both values are real numbers (no NaN/inf)
mask_valid = (
    ancestral_derived_data["ancestral"].notna() &
    ancestral_derived_data["derived"].notna() &
    np.isfinite(ancestral_derived_data["ancestral"]) &
    np.isfinite(ancestral_derived_data["derived"])
)


ancestral_derived_data = ancestral_derived_data.dropna(subset=['ancestral', 'derived'])

# Remove +inf or -inf
ancestral_derived_data = ancestral_derived_data[~ancestral_derived_data.isin([np.inf, -np.inf]).any(axis=1)]
#ancestral_derived_data = ancestral_derived_data.sample(n=100000, random_state=42)



In [ ]:
ancestral_derived_data

In [ ]:
x = ancestral_derived_data['ancestral'].values
y = ancestral_derived_data['derived'].values
colors = ancestral_derived_data['differential_activity'].values

r, p = pearsonr(x, y)

print("Pearson r:", r)
print("p-value:", p)

# Create the KDE (Kernel Density Estimate)
#values = np.vstack([x, y])
#kernel = gaussian_kde(values)

# Evaluate the KDE for each data point
#density = kernel(values)

#max_density_threshold = 0.1 

# Density maximum value clipping 
#density_capped = np.clip(density, a_min=None, a_max=max_density_threshold)

In [ ]:
hb = plt.hexbin(
    x[(x<100)&(y<100)], y[(x<100)&(y<100)],
    gridsize=200,          # adjust for coarser/finer grid
    cmap=custom_cmap_light,
    mincnt=1,             # only show hexes with at least 1 point
    bins='log'            # color ~ log(count); remove if you want raw counts
    # alpha=0.9           # optional, usually you can skip alpha for hexbin
)

plt.xlabel(r'Ancestral allele mad score')
plt.ylabel(r'Derived allele mad score')
plt.xlim(0, 100)
plt.ylim(0,100)
xticks = plt.xticks()[0]
yticks = plt.yticks()[0]
plt.xticks([xticks[0], xticks[-1]])
plt.yticks([yticks[0], yticks[-1]])

const.save_fig(plt, f'correlation_between_alleles_min_dna_{min_DNA_counts}', output_path)
plt.show()


In [ ]:
mask = (x < 20) & (y < 20)

r, p = pearsonr(x[mask], y[mask])

hb = plt.hexbin(
    x[mask], y[mask],
    gridsize=200,
    cmap=custom_cmap_light,
    mincnt=1,
    bins='log'
)

plt.xlabel(r'Ancestral allele mad score')
plt.ylabel(r'Derived allele mad score')
plt.xlim(-5, 20)
plt.ylim(-5, 20)

xticks = plt.xticks()[0]
yticks = plt.yticks()[0]
plt.xticks([xticks[0], xticks[-1]])
plt.yticks([yticks[0], yticks[-1]])

# annotate Pearson r and p-value
plt.text(
    0.03, 0.97,
    f"Pearson r = {r:.3f}\np = {p:.2e}\nN = {mask.sum()}",
    transform=plt.gca().transAxes,
    va="top", ha="left"
)

const.save_fig(plt, f'correlation_between_alleles_min_dna_{min_DNA_counts}', output_path)
plt.show()

In [ ]:
plt.clf()

fig, ax = plt.subplots()

mask_true = pd.Series(colors).eq(True).to_numpy()
mask_false = pd.Series(colors).eq(False).to_numpy()
mask_na = pd.Series(colors).isna().to_numpy()

ax.scatter(x[mask_na], y[mask_na], c='lightgray', alpha=0.1, s=30, edgecolors='darkgray', label='diff active = NA')
ax.scatter(x[mask_false], y[mask_false], c='orange', alpha=0.1, s=30, edgecolors='darkorange', label='diff active = False')
ax.scatter(x[mask_true], y[mask_true], c='red', alpha=0.1, s=30, edgecolors='darkred', label='diff active = True')

line_min = max(ax.get_xlim()[0], ax.get_ylim()[0])
line_max = min(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([line_min, line_max], [line_min, line_max], linestyle='--', color='black', linewidth=1)

ax.set_xlabel(r'Ancestral sequence activity (MAD)')
ax.set_ylabel(r'Derived sequence activity (MAD)')

ax.set_xlim(-5, 20)
ax.set_ylim(-5, 20)

xticks = ax.get_xticks()
yticks = ax.get_yticks()
ax.set_xticks([xticks[0], xticks[-1]])
ax.set_yticks([yticks[0], yticks[-1]])

ax.legend(frameon=False, fontsize=10)

const.save_fig(plt, f'diff_active_oligos_allele_scatter_min_dna_{min_DNA_counts}', output_path)
plt.show()

### Calc Pearson's corr between ancstral and derived (all sequences)

In [ ]:
mask = (
    np.isfinite(ancestral_derived_data["ancestral"]) &
    np.isfinite(ancestral_derived_data["derived"])
)

r, p = pearsonr(
    ancestral_derived_data.loc[mask, "ancestral"],
    ancestral_derived_data.loc[mask, "derived"]
)

print(f"Pearson r = {r:.4f}")
print(f"p-value = {p:.3e}")

In [ ]:
plt.clf()

fig, ax = plt.subplots()
cap = 20

ancestral_derived_data_capped = ancestral_derived_data[(ancestral_derived_data['ancestral']<cap) & (ancestral_derived_data['derived']<cap)].copy()
x_capped = ancestral_derived_data_capped['ancestral'].values
y_capped = ancestral_derived_data_capped['derived'].values
colors_capped = ancestral_derived_data_capped['differential_activity'].values


# Separate the data by activity status
mask_diff = pd.Series(colors_capped).eq(True).to_numpy()
mask_active = pd.Series(colors_capped).eq(False).to_numpy()
mask_non_active = pd.Series(colors_capped).isna().to_numpy()


from matplotlib.colors import LinearSegmentedColormap

gray_cmap = LinearSegmentedColormap.from_list(
    "gray_cmap",
    ["#DAD5D5", "#DAD5D5"],
    N=256
)

orange_cmap = LinearSegmentedColormap.from_list(
    "orange_cmap",
    ["#D87A1C", "#D87A1C"],
    N=256
)

red_cmap = LinearSegmentedColormap.from_list(
    "red_cmap",
    ["#D73027", "#D73027"],
    N=256
)

# Hexbin for active (differential)
hb_active = ax.hexbin(
    x_capped[mask_non_active], y_capped[mask_non_active],
    gridsize=200,
    cmap=gray_cmap,
    mincnt=1,
    alpha=1,
    #linewidths=0,
    label='diff active = True'
)

# Hexbin for active regular
hb_non_active = ax.hexbin(
    x_capped[mask_active], y_capped[mask_active],
    gridsize=140,
    cmap=red_cmap,
    mincnt=1,
    alpha=0.5,
    linewidths=0,
    label='diff active = False'
)


# Scatter with hexagon markers for differential active
ax.scatter(
    x_capped[mask_diff], y_capped[mask_diff],
    c='indigo',
    #marker='h',
    s=7,
    alpha=0.3,
    edgecolors='none',
    linewidths=0.5,
    label='diff active = True (overlay)'
)



line_min = max(ax.get_xlim()[0], ax.get_ylim()[0])
line_max = min(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([line_min, line_max], [line_min, line_max], linestyle='--', color='black', linewidth=1)

ax.set_xlabel(r'Ancestral sequence activity (MAD)')
ax.set_ylabel(r'Derived sequence activity (MAD)')

ax.set_xlim(-5, 20)
ax.set_ylim(-5, 20)

xticks = ax.get_xticks()
yticks = ax.get_yticks()
ax.set_xticks([xticks[0], xticks[-1]])
ax.set_yticks([yticks[0], yticks[-1]])

ax.legend(frameon=False, fontsize=10)

const.save_fig(plt, f'diff_active_oligos_allele_scatter_min_dna_{min_DNA_counts}', output_path)
plt.show()

## Overlap with endogenous signal

### Load annotations

In [ ]:

annotations = pd.read_csv(f'/home/labs/davidgo/Collaboration/humanMPRA/top_candidates/chondrocytes/humanMPRA_annotations_v3.csv', 
                     header=0)
annotations = annotations.drop_duplicates(subset=["oligo"], keep = "first") #There are several HH oligos which are duplicated
annotations = annotations[#~(annotations['orientation_fix']=='fixed_in_L4')&
                           ~(annotations['oligo'].str.contains('SCREEN_EH'))&
                          ~(annotations['oligo'].str.contains('hh.missing.oligos')) ]

#Filter for minimum DNA counts
print(len(annotations))
annotations = annotations[(annotations['DNA_counts_raw_derived']>min_DNA_counts)]
print(len(annotations))


# create BED of annotations
annotations_bed_df = annotations[["chromosome","start","end","oligo"]]
annotations_bed_df.loc[:,"start"] = annotations_bed_df["start"].map(int) # already zero-based - Make sure with Simon!!
annotations_bed_df.loc[:,"end"] = annotations_bed_df["end"].map(int)
annotations_bed_df=annotations_bed_df.sort_values(['chromosome','start'], ascending = [True, True])
annotations_bed = pybedtools.BedTool.from_dataframe(annotations_bed_df)

#### Double check that there are equal number of cCREs in annoatations and the original activity df

In [ ]:
def remove_derivation(s):
    # remove "_derived_*" OR "_ancestral_*"
    return re.sub(r'_derived[^_]*', '', re.sub(r'_ancestral[^_]*', '', s))

full_norm = full_activity_df["oligo"].apply(remove_derivation)
ann_norm  = annotations["oligo"].apply(remove_derivation)
set_full = set(full_norm)
set_ann  = set(ann_norm)

missing_in_annotations = set_full - set_ann
missing_in_full = set_ann - set_full

df_missing_in_annotations = full_activity_df[full_norm.isin(missing_in_annotations)]
df_missing_in_full        = annotations[ann_norm.isin(missing_in_full)]

print("len(full_activity_df) =", len(full_activity_df))
print("len(annotations)      =", len(annotations))

print("unique normalized in full =", len(set_full))
print("unique normalized in ann  =", len(set_ann))
print("sets equal?               =", set_full == set_ann)


## Load SCREEN database for annotation

In [ ]:
screen_version='V2'

screen_file_hg19 = '/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Chromatin/ENCODE SCREEN/' + screen_version + '/GRCh37-lifted-cCREs.bed'
screen_annotation_file='/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Chromatin/ENCODE SCREEN/' + screen_version + '/GRCh38-cCREs.bed'

# the annotations were truncated in the liftover, need to join with the hg38 file
screen_hg19_df=pd.read_csv(screen_file_hg19, sep='\t',header=None)
screen_annotation_df=pd.read_csv(screen_annotation_file, sep='\t',header=None)
screen_annotation_df.drop(columns=[0,1,2,3], inplace=True)
print(len(screen_hg19_df))
screen_df = pd.merge(screen_hg19_df,screen_annotation_df,left_on=4,right_on=4,how='left')
print(len(screen_df))

screen_df.drop(columns=['5_x'], inplace=True)
screen_df.rename(columns={"5_y": "regulatoryClass"},inplace=True)
screen_df["regulatoryClass"] = screen_df["regulatoryClass"].str.replace(',CTCF-bound','')
screen_df.drop(screen_df[screen_df.regulatoryClass =='CTCF-only'].index, inplace=True)
screen_df.drop(columns = (3),inplace=True)
screen_df.rename(columns = {4:'enhancerID'},inplace=True)


screen_df = screen_df.sort_values([0,1], ascending = [True, True])
screen_bed=pybedtools.BedTool.from_dataframe(screen_df) 

In [ ]:

merged_screen_bed = screen_bed.merge()

# Intersect with annotations
intersect_bed = annotations_bed.intersect(merged_screen_bed, wao=True)
intersect_df = intersect_bed.to_dataframe()
intersect_df.drop(['score','strand','thickStart'], axis=1,inplace=True)

print('#overlpas with SCREEN elements:', len(intersect_df))

# Thickend represents the #BP overlap between each oligo and a SCREEN element
intersect_df = (intersect_df.groupby([col for col in intersect_df.columns if col != 'thickEnd'], as_index=False)
      .agg({'thickEnd': 'sum'})
)

print('# overlaps after collpasing per oligo :', len(intersect_df))

oligos_SCREEN = pd.merge(annotations, intersect_df[['thickEnd', 'name']], left_on='oligo', right_on = "name", how='left')
oligos_SCREEN.drop("name", axis=1,inplace=True)
   

In [ ]:
# grab classes per oligo
classes_df = annotations_bed.intersect(screen_bed, wao=True).to_dataframe()
classes_df = classes_df[['name', 'itemRgb']]
classes_df = classes_df.groupby('name', as_index=False).agg({'itemRgb': list})
classes_df.rename(columns={'itemRgb': 'CRE_class'}, inplace=True)


# join with main
oligos_class = pd.merge(annotations, classes_df[['CRE_class', 'name']], left_on='oligo', right_on = "name", how='left')
oligos_class.drop(columns = ["name"],inplace=True)
oligos_class['CRE_class'] = oligos_class['CRE_class'].apply(lambda x: ', '.join(map(str, x)))


oligos_class['SCREEN2_class_strict'] = (
    oligos_class['CRE_class']
    .apply(lambda x: ', '.join(map(str, x)) if isinstance(x, list) else x)  # Convert list to string if needed
    .apply(lambda x: set(x.split(', ')) if isinstance(x, str) else set())  # Convert to set for uniqueness
    .apply(lambda x: next(iter(x)) if len(x) == 1 else 'more than one')    # Extract single value or mark as "more than one"
)

In [ ]:
# Ensure flag exists
oligos_class.loc[:, "in_ATAC-seq_peak"] = (oligos_class["ATACseq_peaks_fetal_chondrocytes"].abs() < 1)

# Create combined category: pELS / dELS / PLS × in/out of ATAC-seq
oligos_class["category"] = np.select(
    [
        (oligos_class["SCREEN2_class_strict"] == "pELS") & (oligos_class["in_ATAC-seq_peak"]),
        (oligos_class["SCREEN2_class_strict"] == "dELS") & (oligos_class["in_ATAC-seq_peak"]),
        (oligos_class["SCREEN2_class_strict"] == "PLS") & (oligos_class["in_ATAC-seq_peak"]),
        (oligos_class["SCREEN2_class_strict"] == "pELS") & (~oligos_class["in_ATAC-seq_peak"]),
        (oligos_class["SCREEN2_class_strict"] == "dELS") & (~oligos_class["in_ATAC-seq_peak"]),
        (oligos_class["SCREEN2_class_strict"] == "PLS") & (~oligos_class["in_ATAC-seq_peak"]),
    ],
    [
        "proximal enhancer-like + ATAC-seq peak",
        "distal enhancer-like + ATAC-seq peak",
        "promoter-like + ATAC-seq peak",
        "proximal enhancer-like",
        "distal enhancer-like",
        "promoter-like",
    ],
    default="Other"
)

# Filter x-range
to_plot = oligos_class.loc[oligos_class["normalized_activity_estimate_derived"].between(-4, 10)].copy()

# Define colors and styles
color_map = {
    "PLS": "green",
    "pELS": "purple",
    "dELS": "red"
}
style_map = {
    True: "--",   # in ATAC-seq
    False: "-"    # outside ATAC-seq
}

plt.figure(figsize=(8, 6))

for cls in ["PLS", "pELS", "dELS"]:
    for in_peak in [True, False]:
        subset = to_plot[
            (to_plot["SCREEN2_class_strict"] == cls) &
            (to_plot["in_ATAC-seq_peak"] == in_peak)
        ].dropna(subset=["normalized_activity_estimate_derived"])
        if subset.empty:
            continue
        sns.ecdfplot(
            data=subset,
            x="normalized_activity_estimate_derived",
            stat="proportion",
            color=color_map[cls],
            linestyle=style_map[in_peak],
            label=f"{cls} {'in' if in_peak else 'outside'} ATAC-seq (n={len(subset)})"
        )

plt.xlabel("Activity statistic of derived allele")
plt.ylabel("Empirical Cumulative Distribution Function")
plt.legend(fontsize=10)
plt.tight_layout()

const.save_fig(plt, 'derived_activity_by_ATAC_and_SCREEN2_strict_styled_min_dna_{min_DNA_counts}', output_path)
plt.show()

# ---- Kolmogorov–Smirnov tests between all pairs ----
print("\n### Kolmogorov–Smirnov Tests Between All Pairs ###\n")

# Collect all group distributions
group_values = {}
for cat in to_plot["category"].unique():
    vals = to_plot.loc[to_plot["category"] == cat, "normalized_activity_estimate_derived"].dropna().to_numpy()
    if vals.size > 0:
        group_values[cat] = vals

# Compute pairwise KS tests
for (name1, vals1), (name2, vals2) in itertools.combinations(group_values.items(), 2):
    if len(vals1) == 0 or len(vals2) == 0:
        continue
    ks = stats.ks_2samp(vals1, vals2)
    p_raw = float(ks.pvalue)
    p_report = p_raw if p_raw > 0 else np.nextafter(0, 1)
    print(f"{name1}  vs  {name2}:  D = {ks.statistic:.4f},  p = {p_report:.3e}")
    
# 1. Pooled ATAC-seq vs. outside ATAC-seq
vals_in_atac = to_plot.loc[to_plot["in_ATAC-seq_peak"], "normalized_activity_estimate_derived"].dropna()
vals_out_atac = to_plot.loc[~to_plot["in_ATAC-seq_peak"], "normalized_activity_estimate_derived"].dropna()
if len(vals_in_atac) and len(vals_out_atac):
    ks = stats.ks_2samp(vals_in_atac, vals_out_atac)
    print(f"Pooled ATAC-seq vs outside ATAC-seq:  D = {ks.statistic:.4f},  p = {ks.pvalue:.3e}")

    
# 2. Pooled enhancers (pELS + dELS) vs promoters (PLS)
vals_enhancers = to_plot.loc[to_plot["SCREEN2_class_strict"].isin(["pELS", "dELS"]),
                             "normalized_activity_estimate_derived"].dropna()
vals_promoters = to_plot.loc[to_plot["SCREEN2_class_strict"] == "PLS",
                             "normalized_activity_estimate_derived"].dropna()
if len(vals_enhancers) and len(vals_promoters):
    ks = stats.ks_2samp(vals_enhancers, vals_promoters)
    print(f"Pooled enhancers (pELS + dELS) vs promoters (PLS):  D = {ks.statistic:.4f},  p = {ks.pvalue:.3e}")

# 3. Pooled proximal vs distal enhancers
vals_proximal = to_plot.loc[to_plot["SCREEN2_class_strict"] == "pELS",
                            "normalized_activity_estimate_derived"].dropna()
vals_distal = to_plot.loc[to_plot["SCREEN2_class_strict"] == "dELS",
                          "normalized_activity_estimate_derived"].dropna()
if len(vals_proximal) and len(vals_distal):
    ks = stats.ks_2samp(vals_proximal, vals_distal)
    print(f"Pooled proximal (pELS) vs distal (dELS) enhancers:  D = {ks.statistic:.4f},  p = {ks.pvalue:.3e}")



In [ ]:
# Ensure flag exists
oligos_class.loc[:, "in_ATAC-seq_peak"] = (oligos_class["ATACseq_peaks_fetal_chondrocytes"].abs() < 1)

# Create combined category: pELS / dELS / PLS × in/out of ATAC-seq
oligos_class["category"] = np.select(
    [
        (oligos_class["SCREEN2_class_strict"] == "pELS") & (oligos_class["in_ATAC-seq_peak"]),
        (oligos_class["SCREEN2_class_strict"] == "dELS") & (oligos_class["in_ATAC-seq_peak"]),
        (oligos_class["SCREEN2_class_strict"] == "PLS") & (oligos_class["in_ATAC-seq_peak"]),
        (oligos_class["SCREEN2_class_strict"] == "pELS") & (~oligos_class["in_ATAC-seq_peak"]),
        (oligos_class["SCREEN2_class_strict"] == "dELS") & (~oligos_class["in_ATAC-seq_peak"]),
        (oligos_class["SCREEN2_class_strict"] == "PLS") & (~oligos_class["in_ATAC-seq_peak"]),
    ],
    [
        "proximal enhancer-like + ATAC-seq peak",
        "distal enhancer-like + ATAC-seq peak",
        "promoter-like + ATAC-seq peak",
        "proximal enhancer-like",
        "distal enhancer-like",
        "promoter-like",
    ],
    default="Other"
)

# Filter x-range
to_plot = oligos_class.loc[oligos_class["normalized_activity_estimate_derived"].between(-4, 10)].copy()


# ---- Legend order (edit as you like) ----
legend_order = [
    "proximal enhancer-like",
    "distal enhancer-like",
    "promoter-like",
    "proximal enhancer-like + ATAC-seq peak",
    "distal enhancer-like + ATAC-seq peak",
    "promoter-like + ATAC-seq peak"
]

# ---- Visual styling ----
lw_in_peak      = 3   # thicker
alpha_in_peak   = 1.0
lw_outside_peak = 2.0
alpha_outside   = 0.55  # more transparent

color_map = {"PLS": "green", "pELS": "purple", "dELS": "red"}

plt.figure(figsize=(6, 6))
ax = plt.gca()

# store one handle per category so legend is clean
handle_by_category = {}
n_by_category = {}

for cls in ["PLS", "pELS", "dELS"]:
    for in_peak in [True, False]:
        subset = to_plot[
            (to_plot["SCREEN2_class_strict"] == cls) &
            (to_plot["in_ATAC-seq_peak"] == in_peak)
        ].dropna(subset=["normalized_activity_estimate_derived"])

        if subset.empty:
            continue

        category_text = subset["category"].iloc[0]

        # Styling: both solid; in-peak thicker
        lw = lw_in_peak if in_peak else lw_outside_peak
        alpha = alpha_in_peak if in_peak else alpha_outside

        sns.ecdfplot(
            data=subset,
            x="normalized_activity_estimate_derived",
            stat="proportion",
            color=color_map[cls],
            linestyle="-",      # <-- force solid everywhere
            linewidth=lw,
            alpha=alpha,
            label=None,         # build legend manually
            ax=ax
        )

        # Save handle for legend (first time we see this category)
        if category_text not in handle_by_category:
            handle_by_category[category_text] = ax.lines[-1]
            n_by_category[category_text] = len(subset)
        else:
            # optional: if you want n to reflect total across both in/out, accumulate
            n_by_category[category_text] += len(subset)

plt.xlabel("Activity statistic of derived allele")
plt.ylabel("Empirical Cumulative\nDistribution Function")

xticks = plt.xticks()[0]
yticks = plt.yticks()[0]
plt.xticks([xticks[0],0, xticks[-1]])
plt.yticks([yticks[0], yticks[-1]])

# ---- Ordered legend ----
ordered_categories = [c for c in legend_order if c in handle_by_category]
ordered_categories += [c for c in handle_by_category if c not in ordered_categories]

handles = [handle_by_category[c] for c in ordered_categories]
#labels  = [f"{c} (N={n_by_category[c]})" for c in ordered_categories]  # or just `c`
labels  = [f"{c}" for c in ordered_categories]  # or just `c`

ax.legend(handles, labels, fontsize=10, title="")
plt.tight_layout()

const.save_fig(plt, 'derived_activity_by_ATAC_and_SCREEN2_strict_styled', output_path)
plt.show()



# ---- Kolmogorov–Smirnov tests between all pairs ----
print("\n### Kolmogorov–Smirnov Tests Between All Pairs ###\n")

# Collect all group distributions
group_values = {}
for cat in to_plot["category"].unique():
    vals = to_plot.loc[to_plot["category"] == cat, "normalized_activity_estimate_derived"].dropna().to_numpy()
    if vals.size > 0:
        group_values[cat] = vals

# Compute pairwise KS tests
for (name1, vals1), (name2, vals2) in itertools.combinations(group_values.items(), 2):
    if len(vals1) == 0 or len(vals2) == 0:
        continue
    ks = stats.ks_2samp(vals1, vals2)
    p_raw = float(ks.pvalue)
    p_report = p_raw if p_raw > 0 else np.nextafter(0, 1)
    print(f"{name1}  vs  {name2}:  D = {ks.statistic:.4f},  p = {p_report:.3e}")
    
# 1. Pooled ATAC-seq vs. outside ATAC-seq
vals_in_atac = to_plot.loc[to_plot["in_ATAC-seq_peak"], "normalized_activity_estimate_derived"].dropna()
vals_out_atac = to_plot.loc[~to_plot["in_ATAC-seq_peak"], "normalized_activity_estimate_derived"].dropna()
if len(vals_in_atac) and len(vals_out_atac):
    ks = stats.ks_2samp(vals_in_atac, vals_out_atac)
    print(f"Pooled ATAC-seq vs outside ATAC-seq:  D = {ks.statistic:.4f},  p = {ks.pvalue:.3e}")

    
# 2. Pooled enhancers (pELS + dELS) vs promoters (PLS)
vals_enhancers = to_plot.loc[to_plot["SCREEN2_class_strict"].isin(["pELS", "dELS"]),
                             "normalized_activity_estimate_derived"].dropna()
vals_promoters = to_plot.loc[to_plot["SCREEN2_class_strict"] == "PLS",
                             "normalized_activity_estimate_derived"].dropna()
if len(vals_enhancers) and len(vals_promoters):
    ks = stats.ks_2samp(vals_enhancers, vals_promoters)
    print(f"Pooled enhancers (pELS + dELS) vs promoters (PLS):  D = {ks.statistic:.4f},  p = {ks.pvalue:.3e}")

# 3. Pooled proximal vs distal enhancers
vals_proximal = to_plot.loc[to_plot["SCREEN2_class_strict"] == "pELS",
                            "normalized_activity_estimate_derived"].dropna()
vals_distal = to_plot.loc[to_plot["SCREEN2_class_strict"] == "dELS",
                          "normalized_activity_estimate_derived"].dropna()
if len(vals_proximal) and len(vals_distal):
    ks = stats.ks_2samp(vals_proximal, vals_distal)
    print(f"Pooled proximal (pELS) vs distal (dELS) enhancers:  D = {ks.statistic:.4f},  p = {ks.pvalue:.3e}")





# Comparing several cell types

In [ ]:
chondrocytes_SCREEN = '/in_vitro_differentiated_chondrocytes/ENCFF807AUZ_ENCFF466YVQ_ENCFF317LGP_ENCFF044ORH.bed'
osteocytes_SCREEN = '/in_vitro_differentiated_osteocytes/ENCFF226FAT_ENCFF582GHH_ENCFF441MGU_ENCFF897TLT.bed'
all_cell_types_SCREEN = '/all_cell_types/GRCh38-cCREs.bed'

In [ ]:
# Filter for promoters only (PLS) from oligo_class
promoter_data = oligos_class[oligos_class['SCREEN2_class_strict'] == 'PLS'].copy()

# Create a label for plotting
promoter_data['ATAC_status'] = promoter_data['in_ATAC-seq_peak'].map({
    True: f'In ATAC-seq peak',
    False: 'Not in ATAC-seq peak'
})

promoter_data

# Remove NaN and infinite values
promoter_data = promoter_data[
    promoter_data['normalized_activity_estimate_derived'].notna() &
    np.isfinite(promoter_data['normalized_activity_estimate_derived'])
]

promoter_data = promoter_data[
    promoter_data['normalized_activity_estimate_derived'].between(-4, 10)
]

# Perform statistical test
vals_pls_in = promoter_data[promoter_data['in_ATAC-seq_peak']]['normalized_activity_estimate_derived']
vals_pls_out = promoter_data[~promoter_data['in_ATAC-seq_peak']]['normalized_activity_estimate_derived']

# Create the swarm plot
plt.figure(figsize=(8, 6))

ax = sns.violinplot(
    data=promoter_data,
    x="ATAC_status",
    y="normalized_activity_estimate_derived",
    order=["In ATAC-seq peak", "Not in ATAC-seq peak"],
    palette=['#66c2a5', "#96dada"],
    inner=None,
    linewidth=0,
    cut=0,
)

ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)

# set custom x tick labels (this is what you wanted from labels=...)
ax.set_xticklabels([
    f"Open Promoter\n(N={len(vals_pls_in)})",
    f"Close Promoter\n(N={len(vals_pls_out)})"
])

yticks = plt.yticks()[0]
ax.set_yticks([yticks[0],0, 10])

ks = ax.get_xaxis().set_tick_params(rotation=0)

ax.set_xlabel('')
ax.set_ylabel('Activity statistic of derived allele')
#ax.set_title('Promoter Activity: In vs Out of ATAC-seq Peaks')

if len(vals_pls_in) > 0 and len(vals_pls_out) > 0:
    ks_stat, ks_pval = stats.ks_2samp(vals_pls_in, vals_pls_out)
    plt.text(
        0.98, 0.98,
        f'KS test:\nD = {ks_stat:.4f}\np = {ks_pval:.3e}\nN(in) = {len(vals_pls_in)}\nN(out) = {len(vals_pls_out)}',
        transform=plt.gca().transAxes,
        ha='right', va='top',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8)
    )

plt.tight_layout()
#const.save_fig(plt, f'promoter_swarmplot_ATAC_comparison_min_dna_{min_DNA_counts}', output_path)
plt.show()

print(f"Promoters in ATAC-seq peaks: {len(vals_pls_in)}")
print(f"Promoters not in ATAC-seq peaks: {len(vals_pls_out)}")

In [ ]:
# Filter for enhancers (pELS + dELS) and promoters (PLS) that are in ATAC-seq peaks
enhancer_data = oligos_class[
    (oligos_class['SCREEN2_class_strict'].isin(['pELS', 'dELS'])) &
    (oligos_class['in_ATAC-seq_peak'])
].copy()

promoter_data = oligos_class[
    (oligos_class['SCREEN2_class_strict'] == 'PLS') &
    (oligos_class['in_ATAC-seq_peak'])
].copy()

# Remove NaN and infinite values
enhancer_data = enhancer_data[
    enhancer_data['normalized_activity_estimate_derived'].notna() &
    np.isfinite(enhancer_data['normalized_activity_estimate_derived'])
]

promoter_data = promoter_data[
    promoter_data['normalized_activity_estimate_derived'].notna() &
    np.isfinite(promoter_data['normalized_activity_estimate_derived'])
]

# Filter for activity range
enhancer_data = enhancer_data[
    enhancer_data['normalized_activity_estimate_derived'].between(-4, 10)
]

promoter_data = promoter_data[
    promoter_data['normalized_activity_estimate_derived'].between(-4, 10)
]

# Extract values
vals_enhancers = enhancer_data['normalized_activity_estimate_derived']
vals_promoters = promoter_data['normalized_activity_estimate_derived']

# Create the violin plot
plt.figure(figsize=(8, 6))

ax = sns.violinplot(
    data=pd.DataFrame({
        'Activity': pd.concat([vals_enhancers, vals_promoters]),
        'Type': ['Open enhancers'] * len(vals_enhancers) + ['Open promoters'] * len(vals_promoters)
    }),
    x="Type",
    y="Activity",
    order=["Open promoters","Open enhancers"],
    palette=['#66c2a5', "#d86dc1"],
    inner=None,
    linewidth=0,
    cut=0,
)

ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)

ax.set_xticklabels([
    f"Open promoters\n(N={len(vals_promoters)})",
    f"Open enhancers\n(N={len(vals_enhancers)})"
])

yticks = plt.yticks()[0]
ax.set_yticks([yticks[0],0, 10])

ax.set_xlabel('')
ax.set_ylabel('Activity statistic of derived allele')

plt.tight_layout()
plt.show()

# Statistical test
if len(vals_enhancers) > 0 and len(vals_promoters) > 0:
    ks_stat, ks_pval = stats.ks_2samp(vals_enhancers, vals_promoters)
    print(f"KS test:")
    print(f"D = {ks_stat:.4f}")
    print(f"p = {ks_pval:.3e}")
    print(f"N(enhancers) = {len(vals_enhancers)}")
    print(f"N(promoters) = {len(vals_promoters)}")

## Comparison to AI predictions

In [ ]:
l1_df = pd.read_csv('/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes/QC_pipeline/input/L1-3/L1.fasta.all.tsv', sep='\t',header=None)
l2_df = pd.read_csv('/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes/QC_pipeline/input/L1-3/L2.fasta.all.tsv', sep='\t',header=None)
l3_df = pd.read_csv('/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes/QC_pipeline/input/L1-3/L3.fasta.all.tsv', sep='\t',header=None)

In [ ]:
# stack the three prediction tables together
# they all share the same column layout, so a simple vertical concat is fine
l123_df = pd.concat([l1_df, l2_df, l3_df], ignore_index=True)
l123_df.rename(columns={0: 'oligo'}, inplace=True)

print("combined L1‑L2‑L3 shape:", l123_df.shape)
l123_df.head()

In [ ]:
# compute mean/median across the numeric prediction columns
# column 0 is the sequence identifier, all the rest are floats
num_cols = l123_df.columns.drop('oligo')


l123_df['mean'] = l123_df[num_cols].mean(axis=1)
l123_df['median'] = l123_df[num_cols].median(axis=1)

l123_df['status'] = l123_df['oligo'].str.extract(r'_(ancestral|derived)_', expand=False)
print(len(l123_df))
l123_df = l123_df[l123_df['status'].notna()]  # keep only rows where we could determine status  
print(len(l123_df))

In [ ]:
full_activity_df = pd.read_csv('/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes/quantitative_analysis_combined/comb_df_combined_fdr.csv')


In [ ]:
merged_AI_MPRA_activity = pd.merge(
    l123_df,
    full_activity_df,
    how='inner',
    on='oligo'
)

In [ ]:
print(len(merged_AI_MPRA_activity))
merged_AI_MPRA_activity = merged_AI_MPRA_activity[~merged_AI_MPRA_activity['oligo'].str.contains('Ctrl', case=False, na=False)]

print(len(merged_AI_MPRA_activity))


In [ ]:
x = merged_AI_MPRA_activity["ratio_log_rep_comb"].to_numpy()
y = merged_AI_MPRA_activity["median"].to_numpy()

mask = (np.abs(x) <= 4) & (np.abs(y) <= 4)

x_f = x[mask]
y_f = y[mask]

fig, ax = plt.subplots()

hb = ax.hexbin(x, y, gridsize=400, mincnt=1,cmap=custom_cmap_bolder)  # increase/decrease gridsize as needed
fig.colorbar(hb, ax=ax, label="Point density")

ax.set_xlabel("Observed log2(RNA/DNA) ")
ax.set_ylabel("Predicted log2(RNA/DNA)")
ax.set_title("median_logFC vs logFC (density)")

plt.show()

In [ ]:
# x and y can be lists / numpy arrays / pandas Series
x = np.asarray(x, dtype=float)
y = np.asarray(y, dtype=float)

# drop NaNs (and infs) pairwise
mask = np.isfinite(x) & np.isfinite(y)
x2, y2 = x[mask], y[mask]

# Pearson (linear)
r, p = pearsonr(x2, y2)
print(f"Pearson r = {r:.4f} (p = {p:.3g}, n = {len(x2)})")

# Spearman (rank-based)
rho, p_s = spearmanr(x2, y2)
print(f"Spearman rho = {rho:.4f} (p = {p_s:.3g}, n = {len(x2)})")

In [ ]:
merged_AI_MPRA_activity 


In [ ]:
merged_AI_MPRA_activity[merged_AI_MPRA_activity['oligo'].str.contains('Ctrl')]

In [ ]:
x = merged_AI_MPRA_activity["ratio_log_rep_comb"].to_numpy()
y = merged_AI_MPRA_activity["median"].to_numpy()

mask = (np.abs(x) <= 4) & (np.abs(y) <= 4)

x_f = x[mask]
y_f = y[mask]

fig, ax = plt.subplots()

hb = ax.hexbin(x, y, gridsize=400, mincnt=1,cmap=custom_cmap_bolder)  # increase/decrease gridsize as needed
fig.colorbar(hb, ax=ax, label="Point density")

ax.set_xlabel("log2(RNA/DNA) ")
ax.set_ylabel("predicted (RNA/DNA)")
ax.set_title("median_logFC vs logFC (density)")

plt.show()

In [ ]:
merged_AI_MPRA_activity

In [ ]:
merged_AI_MPRA_activity

In [ ]:
l123_df['oligo'] = l123_df['oligo'].str.replace(
    r"[_\- ]?(ancestral|derived)",
    "",
    regex=True,
    flags=re.IGNORECASE
)

# 1) keep only the columns we need for pairing
tmp = l123_df[["oligo", "status", "mean", "median"]].copy()

# 2) pivot to get one row per oligo, with separate ancestral/derived columns
paired = (
    tmp.pivot_table(
        index="oligo",
        columns="status",
        values=["mean", "median"],
        aggfunc="first",   # if duplicates exist per (oligo,status), change to np.median / np.mean etc.
    )
)

# 3) flatten multiindex columns: ('median','derived') -> 'median_derived'
paired.columns = [f"{metric}_{status}" for metric, status in paired.columns]
paired = paired.reset_index()

# 4) keep only complete pairs (both ancestral and derived exist)
paired = paired.dropna(subset=["median_ancestral", "median_derived", "mean_ancestral", "mean_derived"])

# 5) ratios (derived / ancestral)
paired["median_logFC"] = paired["median_derived"] / paired["median_ancestral"]
paired["mean_logFC"]   = paired["mean_derived"]   / paired["mean_ancestral"]

# (optional) avoid inf if ancestral is 0
# paired = paired.replace([np.inf, -np.inf], np.nan).dropna(subset=["median_logFC", "mean_logFC"])

print("original rows:", len(l123_df))
print("paired rows:", len(paired))

paired_df = paired

In [ ]:
l123_df

In [ ]:
full_comparative_df = pd.read_csv("/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes/comparative_analysis_combined/mpranalyze_comp_res_fdr_complete.csv")
full_comparative_df = full_comparative_df[['oligo','logFC','differntial.wo_controls']]
full_comparative_df = full_comparative_df.rename(
    columns={'differntial.wo_controls': 'differential_activity'}
)

In [ ]:
merged_AI_MPRA = pd.merge(
    paired_df,
    full_comparative_df,
    how='inner',
    on='oligo'
)

merged_AI_MPRA_diff  = merged_AI_MPRA[merged_AI_MPRA['differential_activity'] == True]

In [ ]:
merged_AI_MPRA

In [ ]:
import matplotlib.pyplot as plt

x = merged_AI_MPRA_diff["logFC"].to_numpy()
y = merged_AI_MPRA_diff["median_logFC"].to_numpy()

mask = (np.abs(x) <= 2) & (np.abs(y) <= 50)

x_f = x[mask]
y_f = y[mask]

fig, ax = plt.subplots()

hb = ax.hexbin(x_f, y_f, gridsize=200, mincnt=1,cmap=custom_cmap_bolder)  # increase/decrease gridsize as needed
fig.colorbar(hb, ax=ax, label="Point density")

ax.set_xlabel("logFC")
ax.set_ylabel("mean_logFC")
ax.set_title("median_logFC vs logFC (density)")

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

x = merged_AI_MPRA["logFC"].to_numpy()
y = merged_AI_MPRA["mean_logFC"].to_numpy()

mask = (np.abs(x) <= 5) & (np.abs(y) <= 3)

x_f = x[mask]
y_f = y[mask]

xy = np.vstack([x_f, y_f])
z = gaussian_kde(xy)(xy)  # density per point

# plot low-density first, high-density last (so dense points appear on top)
idx = np.argsort(z)

fig, ax = plt.subplots()
sc = ax.scatter(x_f[idx], y_f[idx], c=z[idx], s=8, alpha=0.8)
fig.colorbar(sc, ax=ax, label="Local density")

ax.set_xlabel("logFC")
ax.set_ylabel("mean_logFC")
ax.set_title("mean_logFC vs logFC (KDE density coloring)")

plt.show()